# 🚀 Train & Publish Fine-Tuned Qwen2.5-0.5B for ElBruno.LocalLLMs

This notebook trains a **Qwen2.5-0.5B-Instruct** model with **QLoRA** for use with
the [ElBruno.LocalLLMs](https://github.com/elbruno/ElBruno.LocalLLMs) .NET library.

**What it does (end to end):**
1. Installs all Python dependencies (Unsloth, TRL, ONNX Runtime)
2. Downloads training data from the GitHub repo
3. Fine-tunes the model using QLoRA (~15–30 min on a T4 GPU)
4. Merges LoRA adapter weights into the base model
5. Converts the merged model to **ONNX INT4** format
6. Validates the ONNX model with a test prompt
7. Uploads to HuggingFace Hub

**For .NET developers:** You don't need to understand the ML details — just:
1. Click **Runtime → Change runtime type → T4 GPU**
2. Set your `HF_TOKEN` in the Configuration cell
3. Pick a model variant (ToolCalling, RAG, or Instruct)
4. Click **Runtime → Run all**

> **What is QLoRA?** Instead of retraining all 500M parameters, QLoRA freezes the
> model and trains small "adapter" layers (~2% of parameters). This needs way less
> GPU memory and runs fast — think of it like adding a thin specialization layer on
> top of the base model.

## 1️⃣ Install Dependencies

This installs Unsloth (fast QLoRA training), TRL (HuggingFace trainer), and
ONNX Runtime GenAI (model conversion). Takes ~3 minutes.

In [10]:
# Install Unsloth (fast QLoRA training) — includes torch, transformers, peft, trl, datasets
!pip install -q unsloth

# Install additional training dependencies
!pip install -q accelerate bitsandbytes

# Install ONNX Runtime GenAI for model conversion
!pip install -q onnxruntime-genai || echo "⚠️ onnxruntime-genai not available — ONNX conversion will use alternative method"

# Install ONNX and onnx-ir (required by onnxruntime-genai model builder)
!pip install -q onnx onnx-ir

# Install huggingface_hub for uploading
!pip install -q huggingface_hub

print("✅ All dependencies installed.")

✅ All dependencies installed.


## 2️⃣ Configuration

Set your HuggingFace token and choose which model variant to train.

| Variant | Training Data | Best For |
|---------|--------------|----------|
| **ToolCalling** | tool-calling-train.json | Apps that call functions/tools |
| **RAG** | rag-grounded-train.json | Apps that answer from documents |
| **Instruct** | combined-train.json | General purpose (tools + RAG + chat) |

In [11]:
# ╔══════════════════════════════════════════════════════════════╗
# ║  CONFIGURE THESE VALUES                                        ║
# ╚══════════════════════════════════════════════════════════════╝

# Your HuggingFace token (needs write access)
# Option 1: Paste directly (not recommended for shared notebooks)
# HF_TOKEN = "hf_xxxxxxxxxxxxxxxxxxxx"
#
# Option 2: Use Colab secrets (recommended) — add HF_TOKEN in the 🔑 sidebar
try:
    from google.colab import userdata
    HF_TOKEN = userdata.get("HF_TOKEN")
    print("✅ HF_TOKEN loaded from Colab secrets.")
except Exception:
    HF_TOKEN = ""  # ← Paste your token here if not using Colab secrets
    if HF_TOKEN:
        print("✅ HF_TOKEN set manually.")
    else:
        print("⚠️  HF_TOKEN not set — upload step will fail. Add it to Colab secrets (🔑 sidebar) or paste above.")

# Your HuggingFace username
HF_USERNAME = "elbruno"

# Which variant to train: "ToolCalling", "RAG", or "Instruct"
MODEL_VARIANT = "ToolCalling"

# Base model (don't change unless you know what you're doing)
BASE_MODEL = "unsloth/Qwen2.5-0.5B-Instruct"

# ── Derived settings (no need to edit) ────────────────────────────────────
REPO_ID = f"{HF_USERNAME}/Qwen2.5-0.5B-LocalLLMs-{MODEL_VARIANT}"

DATA_FILES = {
    "ToolCalling": "tool-calling-train.json",
    "RAG": "rag-grounded-train.json",
    "Instruct": "combined-train.json",
}
TRAINING_FILE = DATA_FILES[MODEL_VARIANT]

print(f"📦 Model variant : {MODEL_VARIANT}")
print(f"📄 Training data : {TRAINING_FILE}")
print(f"🏷️  HuggingFace repo: {REPO_ID}")
print(f"🧠 Base model    : {BASE_MODEL}")

✅ HF_TOKEN loaded from Colab secrets.
📦 Model variant : ToolCalling
📄 Training data : tool-calling-train.json
🏷️  HuggingFace repo: elbruno/Qwen2.5-0.5B-LocalLLMs-ToolCalling
🧠 Base model    : unsloth/Qwen2.5-0.5B-Instruct


## 3️⃣ Download Training Data

Clones just the `training-data/` folder from the ElBruno.LocalLLMs GitHub repo.

In [12]:
import os

# Sparse checkout — only download the training-data folder and model card template
!git clone --depth 1 --filter=blob:none --sparse \
    https://github.com/elbruno/ElBruno.LocalLLMs.git /content/repo

os.chdir("/content/repo")
!git sparse-checkout set training-data scripts/finetune
os.chdir("/content")

# Verify the training file exists
train_path = f"/content/repo/training-data/{TRAINING_FILE}"
assert os.path.exists(train_path), f"Training file not found: {train_path}"

import json
with open(train_path) as f:
    data = json.load(f)
print(f"✅ Loaded {len(data)} training examples from {TRAINING_FILE}")
print(f"   Example keys: {list(data[0].keys())}")
print(f"   First conversation has {len(data[0]['conversations'])} turns")

fatal: destination path '/content/repo' already exists and is not an empty directory.
✅ Loaded 53 training examples from tool-calling-train.json
   Example keys: ['conversations']
   First conversation has 5 turns


## 4️⃣ Load & Prepare Training Data

The training data is in **ShareGPT format** — each example has a `conversations`
array with turns labeled `system`, `human`, and `gpt`.

We convert this to **ChatML** format, which is what Qwen2.5 natively uses:
```
<|im_start|>system
You are helpful.<|im_end|>
<|im_start|>user
Hello!<|im_end|>
<|im_start|>assistant
Hi there!<|im_end|>
```

In [13]:
from datasets import load_dataset

def format_sharegpt_to_chatml(example):
    """Convert ShareGPT format to ChatML string for training."""
    text_parts = []
    for turn in example["conversations"]:
        role = turn["from"]
        content = turn["value"]
        if role == "system":
            text_parts.append(f"<|im_start|>system\n{content}<|im_end|>")
        elif role == "human":
            text_parts.append(f"<|im_start|>user\n{content}<|im_end|>")
        elif role == "gpt":
            text_parts.append(f"<|im_start|>assistant\n{content}<|im_end|>")
    text = "\n".join(text_parts)
    return {"text": text}

# Load the training data
train_dataset = load_dataset("json", data_files=train_path, split="train")
train_dataset = train_dataset.map(
    format_sharegpt_to_chatml,
    remove_columns=train_dataset.column_names,
)

# Load validation data if available
val_path = "/content/repo/training-data/validation.json"
val_dataset = None
if os.path.exists(val_path):
    val_dataset = load_dataset("json", data_files=val_path, split="train")
    val_dataset = val_dataset.map(
        format_sharegpt_to_chatml,
        remove_columns=val_dataset.column_names,
    )
    print(f"✅ Validation examples: {len(val_dataset)}")

print(f"✅ Training examples: {len(train_dataset)}")
print(f"\n📝 Sample (first 300 chars):\n{train_dataset[0]['text'][:300]}...")

✅ Validation examples: 10
✅ Training examples: 53

📝 Sample (first 300 chars):
<|im_start|>system
You are a helpful assistant.

You have access to the following tools:

[
  {"type": "function", "function": {"name": "get_weather", "description": "Get current weather for a location", "parameters": {"type": "object", "properties": {"location": {"type": "string", "description": "C...


## 5️⃣ Train with QLoRA

This is the main training step. On a free Colab T4 GPU, expect ~15–30 minutes.

**What's happening:**
- The base model loads in 4-bit precision (saves GPU memory)
- QLoRA adds small trainable adapter layers to each attention block
- The trainer runs 3 passes (epochs) over the training data
- Only ~2% of parameters are actually updated

In [14]:
from unsloth import FastLanguageModel
from transformers import TrainingArguments
from trl import SFTTrainer

# ── Hyperparameters (same as scripts/finetune/train_qwen_05b.py) ──
MAX_SEQ_LENGTH = 2048
LORA_R = 16          # LoRA rank — how many parameters the adapter adds
LORA_ALPHA = 32      # Scaling factor (usually 2x rank)
LORA_DROPOUT = 0.05
LEARNING_RATE = 2e-4
BATCH_SIZE = 4
GRAD_ACCUM = 4       # Effective batch size = 4 × 4 = 16
EPOCHS = 3

TARGET_MODULES = [
    "q_proj", "k_proj", "v_proj", "o_proj",
    "gate_proj", "up_proj", "down_proj",
]

# ── Load base model with 4-bit quantization ────────────────────
print(f"Loading {BASE_MODEL} with 4-bit quantization...")
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=BASE_MODEL,
    max_seq_length=MAX_SEQ_LENGTH,
    dtype=None,        # Auto-detect (float16 on T4)
    load_in_4bit=True, # QLoRA: load weights in 4-bit
)
print(f"✅ Model loaded: {model.config._name_or_path}")

# ── Configure QLoRA adapters ───────────────────────────────
model = FastLanguageModel.get_peft_model(
    model,
    r=LORA_R,
    lora_alpha=LORA_ALPHA,
    lora_dropout=LORA_DROPOUT,
    target_modules=TARGET_MODULES,
    bias="none",
    use_gradient_checkpointing="unsloth",  # Saves ~30% VRAM
)
print(f"✅ QLoRA configured: rank={LORA_R}, alpha={LORA_ALPHA}")

Loading unsloth/Qwen2.5-0.5B-Instruct with 4-bit quantization...
==((====))==  Unsloth 2026.3.17: Fast Qwen2 patching. Transformers: 5.3.0.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = None. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

unsloth/qwen2.5-0.5b-instruct-unsloth-bnb-4bit does not have a padding token! Will use pad_token = <|PAD_TOKEN|>.
✅ Model loaded: unsloth/qwen2.5-0.5b-instruct-unsloth-bnb-4bit
✅ QLoRA configured: rank=16, alpha=32


In [15]:
# ── Training arguments ─────────────────────────────────────
OUTPUT_DIR = "/content/output/lora-adapter"

training_args = TrainingArguments(
    output_dir=OUTPUT_DIR,
    num_train_epochs=EPOCHS,
    per_device_train_batch_size=BATCH_SIZE,
    gradient_accumulation_steps=GRAD_ACCUM,
    learning_rate=LEARNING_RATE,
    fp16=True,
    logging_steps=50,
    save_steps=500,
    save_total_limit=2,
    eval_strategy="steps" if val_dataset else "no",
    eval_steps=500 if val_dataset else None,
    warmup_steps=50,
    lr_scheduler_type="cosine",
    optim="paged_adamw_8bit",
    weight_decay=0.01,
    load_best_model_at_end=bool(val_dataset),
    metric_for_best_model="eval_loss" if val_dataset else None,
    report_to="none",
)

# ── Create trainer and start training ──────────────────────
trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    dataset_text_field="text",
    max_seq_length=MAX_SEQ_LENGTH,
    args=training_args,
)

print(f"🚀 Starting training...")
print(f"   Epochs: {EPOCHS}")
print(f"   Batch size: {BATCH_SIZE} (effective: {BATCH_SIZE * GRAD_ACCUM})")
print(f"   Learning rate: {LEARNING_RATE}")
print(f"   This will take ~15-30 min on T4, ~10 min on A100")
print()

trainer.train()

# Save the LoRA adapter
model.save_pretrained(OUTPUT_DIR)
tokenizer.save_pretrained(OUTPUT_DIR)
print(f"\n✅ Training complete! LoRA adapter saved to {OUTPUT_DIR}")

Unsloth: Tokenizing ["text"] (num_proc=6):   0%|          | 0/53 [00:00<?, ? examples/s]

Unsloth: Tokenizing ["text"] (num_proc=6):   0%|          | 0/10 [00:00<?, ? examples/s]

The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'bos_token_id': None}.


🚀 Starting training...
   Epochs: 3
   Batch size: 4 (effective: 16)
   Learning rate: 0.0002
   This will take ~15-30 min on T4, ~10 min on A100



==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 53 | Num Epochs = 3 | Total steps = 12
O^O/ \_/ \    Batch size per device = 4 | Gradient accumulation steps = 4
\        /    Data Parallel GPUs = 1 | Total batch size (4 x 4 x 1) = 16
 "-____-"     Trainable parameters = 8,798,208 of 502,830,976 (1.75% trained)


Step,Training Loss,Validation Loss



✅ Training complete! LoRA adapter saved to /content/output/lora-adapter


## 6️⃣ Merge LoRA Adapter into Base Model

QLoRA only trains small adapter layers. Now we merge those adapters back into
the full model to get a standard HuggingFace checkpoint.

Think of it like: `final_model = base_model + trained_adapter`

In [16]:
MERGED_DIR = "/content/output/merged-model"

# Unsloth provides a convenient merge method
model.save_pretrained_merged(
    MERGED_DIR,
    tokenizer,
    save_method="merged_16bit",  # Full FP16 for ONNX conversion
)

print(f"✅ Merged model saved to {MERGED_DIR}")

# Show output files
for f in sorted(os.listdir(MERGED_DIR)):
    size_mb = os.path.getsize(os.path.join(MERGED_DIR, f)) / (1024 * 1024)
    print(f"  {f:40s} {size_mb:8.1f} MB")

config.json:   0%|          | 0.00/761 [00:00<?, ?B/s]

Found HuggingFace hub cache directory: /root/.cache/huggingface/hub
Checking cache directory for required files...
Cache check failed: model.safetensors not found in local cache.
Not all required files found in cache. Will proceed with downloading.
Checking cache directory for required files...
Cache check failed: tokenizer.model not found in local cache.
Not all required files found in cache. Will proceed with downloading.


Unsloth: Preparing safetensor model files:   0%|          | 0/1 [00:00<?, ?it/s]

model.safetensors:   0%|          | 0.00/988M [00:00<?, ?B/s]

Unsloth: Preparing safetensor model files: 100%|██████████| 1/1 [00:14<00:00, 14.12s/it]


Note: tokenizer.model not found (this is OK for non-SentencePiece models)


Unsloth: Merging weights into 16bit: 100%|██████████| 1/1 [00:05<00:00,  5.78s/it]


Unsloth: Merge process complete. Saved to `/content/output/merged-model`
✅ Merged model saved to /content/output/merged-model
  .cache                                        0.0 MB
  chat_template.jinja                           0.0 MB
  config.json                                   0.0 MB
  model.safetensors                           942.3 MB
  tokenizer.json                               10.9 MB
  tokenizer_config.json                         0.0 MB


## 7️⃣ Convert to ONNX INT4

ElBruno.LocalLLMs uses **ONNX Runtime GenAI** format. This step converts the
merged PyTorch model to an optimized ONNX model with INT4 quantization.

**INT4 means** each weight is stored in 4 bits instead of 16 — the model is ~4x
smaller with only ~1-3% quality loss. A 0.5B model goes from ~1 GB to ~250 MB.

In [17]:
import subprocess, sys

# Ensure ONNX dependencies are available
!pip install -q onnx onnx-ir

ONNX_DIR = "/content/output/onnx-int4"

# Check if onnxruntime-genai is available
try:
    import onnxruntime_genai
    print("✅ onnxruntime-genai available")
except ImportError:
    print("⚠️ onnxruntime-genai not installed. Trying to install...")
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "onnxruntime-genai"], check=False)

print("Converting to ONNX INT4 format...")
print("This takes ~5-10 minutes.\n")

cmd = [
    sys.executable, "-m", "onnxruntime_genai.models.builder",
    "-m", MERGED_DIR,
    "-o", ONNX_DIR,
    "-p", "int4",
    "-e", "cpu",
    "--extra_options", "int4_accuracy_level=4",
]

result = subprocess.run(cmd, capture_output=True, text=True)
print(result.stdout)
if result.returncode != 0:
    print("❌ ONNX conversion FAILED:")
    print(result.stderr)
    raise RuntimeError("ONNX conversion failed")

# Verify expected files exist
expected = ["model.onnx", "genai_config.json", "tokenizer.json", "tokenizer_config.json"]
for fname in expected:
    path = os.path.join(ONNX_DIR, fname)
    assert os.path.exists(path), f"Missing expected file: {fname}"

print("\n📁 ONNX model files:")
total_mb = 0
for f in sorted(os.listdir(ONNX_DIR)):
    size_mb = os.path.getsize(os.path.join(ONNX_DIR, f)) / (1024 * 1024)
    total_mb += size_mb
    print(f"  {f:40s} {size_mb:8.1f} MB")
print(f"  {'TOTAL':40s} {total_mb:8.1f} MB")
print(f"\n✅ ONNX INT4 conversion complete!")

✅ onnxruntime-genai available
Converting to ONNX INT4 format...
This takes ~5-10 minutes.


❌ ONNX conversion FAILED:
Traceback (most recent call last):
  File "<frozen runpy>", line 198, in _run_module_as_main
  File "<frozen runpy>", line 88, in _run_code
  File "/usr/local/lib/python3.12/dist-packages/onnxruntime_genai/models/builder.py", line 17, in <module>
    import onnx_ir as ir
ModuleNotFoundError: No module named 'onnx_ir'



RuntimeError: ONNX conversion failed

## 8️⃣ Validate ONNX Model

Quick sanity check — load the ONNX model and run a test prompt to make sure
it produces reasonable output.

In [ ]:
try:
    import onnxruntime_genai as og
except ImportError:
    print("⚠️ onnxruntime-genai is not available. Skipping validation.")
    print("   The ONNX model files are in:", ONNX_DIR)
    print("   You can validate locally on a machine with onnxruntime-genai installed.")
    og = None

if og is not None:
    print("Loading ONNX model for validation...")
    onnx_model = og.Model(ONNX_DIR)
    onnx_tokenizer = og.Tokenizer(onnx_model)

    # Build a test prompt based on the variant
    if MODEL_VARIANT == "ToolCalling":
        test_prompt = (
            "<|im_start|>system\n"
            "You are a helpful assistant with access to the following tools:\n\n"
            '[{"type":"function","function":{"name":"get_weather","description":"Get current weather",'
            '"parameters":{"type":"object","properties":{"city":{"type":"string"}}}}}]\n\n'
            "When you need to call a tool, respond with a JSON object in this format:\n"
            '{"name": "tool_name", "arguments": {"arg1": "value1"}}\n'
            "<|im_end|>\n"
            "<|im_start|>user\nWhat is the weather in Tokyo?<|im_end|>\n"
            "<|im_start|>assistant\n"
        )
    elif MODEL_VARIANT == "RAG":
        test_prompt = (
            "<|im_start|>system\n"
            "Answer based only on the provided context. Cite sources.\n"
            "<|im_end|>\n"
            "<|im_start|>user\n"
            "Context:\n[1] ONNX Runtime enables fast local inference for ML models.\n\n"
            "Question: What does ONNX Runtime do?<|im_end|>\n"
            "<|im_start|>assistant\n"
        )
    else:  # Instruct
        test_prompt = (
            "<|im_start|>system\nYou are a helpful assistant.<|im_end|>\n"
            "<|im_start|>user\nWhat is machine learning in one sentence?<|im_end|>\n"
            "<|im_start|>assistant\n"
        )

    print(f"\n📝 Test prompt ({MODEL_VARIANT}):\n{test_prompt}")
    print("─" * 60)

    # Generate
    input_tokens = onnx_tokenizer.encode(test_prompt)
    params = og.GeneratorParams(onnx_model)
    params.set_search_options(max_length=300, do_sample=False)
    params.input_ids = input_tokens

    generator = og.Generator(onnx_model, params)
    output_tokens = []
    while not generator.is_done():
        generator.compute_logits()
        generator.generate_next_token()
        token = generator.get_next_tokens()[0]
        output_tokens.append(token)

    output_text = onnx_tokenizer.decode(output_tokens)
    print(f"\n🤖 Model output:\n{output_text}")
    print("─" * 60)

    # Basic sanity checks
    if MODEL_VARIANT == "ToolCalling":
        if "tool_call" in output_text or "get_weather" in output_text:
            print("✅ Model produces tool-calling output!")
        else:
            print("⚠️  Output doesn't contain tool_call tags — may need more training.")
    elif MODEL_VARIANT == "RAG":
        if "ONNX" in output_text or "inference" in output_text.lower():
            print("✅ Model produces grounded output!")
        else:
            print("⚠️  Output may not be well-grounded — check manually.")
    else:
        if len(output_text.strip()) > 10:
            print("✅ Model produces coherent output!")
        else:
            print("⚠️  Output seems too short — check manually.")

    # Clean up to free memory
    del generator, onnx_model, onnx_tokenizer
    print("\n✅ Validation complete.")

## 9️⃣ Upload to HuggingFace

Uploads the ONNX model to HuggingFace Hub with a model card. The repo is created
automatically if it doesn't exist.

In [ ]:
from huggingface_hub import HfApi, create_repo

assert HF_TOKEN, "❌ HF_TOKEN is not set! Go back to the Configuration cell."

api = HfApi(token=HF_TOKEN)

# ── Create repository ──────────────────────────────────────
print(f"Creating/updating repo: {REPO_ID}")
create_repo(REPO_ID, repo_type="model", exist_ok=True, token=HF_TOKEN)
print(f"✅ Repository ready: https://huggingface.co/{REPO_ID}")

# ── Generate model card ────────────────────────────────────
total_mb = sum(
    os.path.getsize(os.path.join(ONNX_DIR, f)) / (1024 * 1024)
    for f in os.listdir(ONNX_DIR)
    if os.path.isfile(os.path.join(ONNX_DIR, f))
)

CAPABILITY_LABELS = {
    "ToolCalling": "tool calling",
    "RAG": "RAG grounded answering",
    "Instruct": "instruction following",
}
capability = CAPABILITY_LABELS[MODEL_VARIANT]
model_name = f"Qwen2.5-0.5B-LocalLLMs-{MODEL_VARIANT}"

# Try to use the template from the repo, fall back to inline
template_path = "/content/repo/scripts/finetune/model-card-template.md"
if os.path.exists(template_path):
    with open(template_path, encoding="utf-8") as f:
        model_card = f.read()
    model_card = model_card.replace("{{MODEL_NAME}}", model_name)
    model_card = model_card.replace("{{REPO_ID}}", REPO_ID)
    model_card = model_card.replace("{{BASE_MODEL}}", "Qwen/Qwen2.5-0.5B-Instruct")
    model_card = model_card.replace("{{CAPABILITY}}", capability)
    model_card = model_card.replace("{{MODEL_SIZE}}", "0.5B")
    model_card = model_card.replace("{{SIZE_MB}}", f"{total_mb:.0f}")
    tags = ["qwen2.5", "onnx", "onnxruntime-genai", "int4", "tool-calling",
            "local-llm", "dotnet", "elbruno", "fine-tuned"]
    model_card = model_card.replace("{{TAGS}}", "\n".join(f"- {t}" for t in tags))
    print("✅ Using model card template from repo.")
else:
    model_card = f"""---
license: apache-2.0
language:
- en
tags:
- qwen2.5
- onnx
- onnxruntime-genai
- int4
- tool-calling
- local-llm
- dotnet
- elbruno
- fine-tuned
base_model: Qwen/Qwen2.5-0.5B-Instruct
---

# {model_name}

Fine-tuned Qwen2.5-0.5B optimized for **{capability}** in [ElBruno.LocalLLMs](https://github.com/elbruno/ElBruno.LocalLLMs).

- **Format:** ONNX INT4 (ONNX Runtime GenAI)
- **Size:** ~{total_mb:.0f} MB
- **Training:** QLoRA rank 16, 3 epochs
- **License:** Apache 2.0
"""
    print("Using inline model card (template not found).")

# Write model card as README.md in the ONNX dir
readme_path = os.path.join(ONNX_DIR, "README.md")
with open(readme_path, "w", encoding="utf-8") as f:
    f.write(model_card)

# ── Upload ─────────────────────────────────────────────────
print(f"\nUploading {total_mb:.0f} MB to {REPO_ID}...")
api.upload_folder(
    folder_path=ONNX_DIR,
    repo_id=REPO_ID,
    repo_type="model",
    commit_message=f"Upload {capability} fine-tuned ONNX INT4 model",
)

print(f"\n🎉 Model published: https://huggingface.co/{REPO_ID}")
print(f"\nYou can now use this model in your .NET app!")

## 🎉 Done! Next Steps

### Use in .NET

Install the NuGet package:
```bash
dotnet add package ElBruno.LocalLLMs
```

Use the fine-tuned model:
```csharp
using ElBruno.LocalLLMs;
using Microsoft.Extensions.AI;

// For tool calling variant
var options = new LocalLLMsOptions
{
    Model = KnownModels.Qwen25_05B_ToolCalling
};

using var client = await LocalChatClient.CreateAsync(options);
var response = await client.GetResponseAsync("Hello!");
```

### Train Other Variants

Re-run this notebook with a different `MODEL_VARIANT`:
- `"ToolCalling"` → Best for apps that call functions/tools
- `"RAG"` → Best for document-grounded Q&A
- `"Instruct"` → Combined training (tools + RAG + chat)

### Resources

- 📦 [ElBruno.LocalLLMs GitHub](https://github.com/elbruno/ElBruno.LocalLLMs)
- 📖 [Fine-tuning Guide](https://github.com/elbruno/ElBruno.LocalLLMs/blob/main/docs/fine-tuning-guide.md)
- 🧪 [Fine-tuned Tool Calling Sample](https://github.com/elbruno/ElBruno.LocalLLMs/tree/main/samples)
- 🤗 [elbruno on HuggingFace](https://huggingface.co/elbruno)